In [2]:
"""
-Arman Bazarchi-
Exploitation Zone — Images notebook

 - Read images from trusted-zone/images/...
 - Only embed images whose UUID exists in metadata_embeddings to cross-check with text data
 - Avoid duplicates by checking existing uuids in image_embeddings collection
 - Store embeddings persistently for similarity search
 - Enrich image embeddings with metadata (species, family, and other columns that we need)
   to have easier access to this data

 - if  we embed images in just 1 folder, all in one , huge data, but 
   later in query we can ask "what life is in this image?" and it must search in 
   huge data and find similars to it.
   However
   if we embed images in seperate collections based on class for example, then 
   in query user must ask "which reptile" or "which mamal" so user must specify the class himself
   then query would be faster because it would only search in that specific class, but
   is not aligned with our aim for this project, as we aim to ask which 'animal is this'

   it would make sense to have them seperately based on kingdom (animalia, plantea, fungi), a user can 
   specifie to this part for sure, but also we are not aiming to store other than animals for this project,
   but would be better choice for a bigger project!

EMBEDDING MODEL: OpenCLIP (ViT-B-32, laion2b_s34b_b79k)
--------------------------------------------------------------------------------
We use OpenCLIP for generating image embeddings to match the query models used in 
task scripts, ensuring dimension compatibility (512-dim embeddings).

Why OpenCLIP for Exploitation Zone:
- Model Consistency: Same model used for embedding and query ensures compatibility
- Fine-Tuning Support: Embeddings work with both baseline and fine-tuned CLIP models
- Multimodal Alignment: CLIP aligns image and text in the same embedding space
- Local Inference: Works completely offline with no API dependencies
- Task Integration: Seamless integration with downstream task scripts

# - removes any temporary file from local storage
"""


from minio import Minio
from PIL import Image
import io, os, re, tempfile
import chromadb
import pandas as pd
from datetime import datetime
from langchain_experimental.open_clip import OpenCLIPEmbeddings  

# -----------------------
#      Configuration
# -----------------------
def process_exploitation_images(
    minio_host="localhost:9000",
    access_key="admin",
    secret_key="password123"):
    """
    Process images from trusted-zone and generate OpenCLIP embeddings.
    
    Args:
        minio_host: MinIO server address
        access_key: MinIO access key
        secret_key: MinIO secret key
    """

    # Initialize MinIO client and verify connection
    client = _initialize_minio_client(minio_host, access_key, secret_key)
    
    # Load metadata CSV for enrichment
    metadata_lookup = _load_metadata(client)
    
    # Initialize ChromaDB collections
    metadata_collection, image_collection = _initialize_chromadb_collections()
    
    # Get existing UUIDs from metadata embeddings
    meta_ids = _get_existing_metadata_ids(metadata_collection)
    
    # Get existing UUIDs from image embeddings
    existing_image_ids = _get_existing_image_ids(image_collection)
    
    # Find candidate images for embedding
    candidates = _find_candidate_images(client, meta_ids, existing_image_ids)
    
    # Process embeddings if candidates exist
    _process_image_embeddings(client, candidates, metadata_lookup, image_collection)
    
    # Display final summary
    _display_final_summary(image_collection)

def _initialize_minio_client(minio_host, access_key, secret_key):
    # Initialize MinIO client and verify trusted zone exists.
    
    trusted_zone = "trusted-zone"
    
    client = Minio(minio_host, access_key=access_key, secret_key=secret_key, secure=False)
    
    # Raise error if trusted zone does not exist
    if not client.bucket_exists(trusted_zone):
        raise SystemExit(f" Trusted zone bucket '{trusted_zone}' does not exist. Cannot continue.")
    
    return client
def _load_metadata(client):
    # Load metadata CSV for enrichment.
    trusted_zone = "trusted-zone"
    
    print(" Loading metadata CSV for enrichment...")
    
    # Find the latest metadata file
    metadata_objs = [
        obj.object_name for obj in client.list_objects(trusted_zone, prefix="metadata/", recursive=True)
        if obj.object_name.lower().endswith(".csv")]
    
    if not metadata_objs:
        raise SystemExit(" No metadata CSV files found in trusted-zone.")
    
    # Use the latest metadata file
    metadata_objs.sort(reverse=True)
    latest_meta = metadata_objs[0]
    print(f" Loading metadata: {latest_meta}")
    
    # Download and load metadata
    resp = client.get_object(trusted_zone, latest_meta)
    data = resp.read()
    resp.close()
    resp.release_conn()
    metadata_df = pd.read_csv(io.BytesIO(data))
    
    # Create metadata lookup dictionary by UUID
    # We take these columns to enrich the metadata of each image embeddings
    metadata_lookup = {}
    for _, row in metadata_df.iterrows():
        uuid_val = row.get('uuid')
        if pd.notna(uuid_val):
            metadata_lookup[uuid_val] = {
                'species': row.get('species', ''),
                'family': row.get('family', ''),
                'class': row.get('class', ''),
                'kingdom': row.get('kingdom', ''),
                'path': row.get('formatted_path', ''),
                'scientific_name': row.get('scientific_name', ''),
                'common': row.get('common', ''),
                'genus': row.get('genus', ''),
                'order': row.get('order', ''),
                'phylum': row.get('phylum', '')
            }
    
    print(f" Loaded metadata for {len(metadata_lookup)} records")
    return metadata_lookup
def _initialize_chromadb_collections():
    # Initialize ChromaDB client and collections.
    # Set the working directory
    
    try:
        script_dir = os.path.dirname(os.path.abspath(__file__))  # in orchestrated.py
    except NameError:
        script_dir = os.getcwd()  # in notebook
        
    chroma_db = os.path.join(script_dir, "../exploitation_db")
    image_collection_name = "image_embeddings"    
    
    # Connect to ChromaDB and collections
    chroma_client = chromadb.PersistentClient(path=chroma_db)
    
    # Metadata embeddings collection 
    metadata_collection = chroma_client.get_or_create_collection(
        name="metadata_embeddings",
        metadata={"description": "Embeddings for trusted metadata records"}
    )
    
    # Image embeddings collection
    image_collection = chroma_client.get_or_create_collection(
        name=image_collection_name,
        metadata={"description": "Embeddings for trusted images"}
    )
    
    return metadata_collection, image_collection
def _get_existing_metadata_ids(metadata_collection):
    # Get existing UUIDs (here its ids) from metadata_embeddings collection.
    
    meta_existing_count = metadata_collection.count()
    meta_ids = []
    
    if meta_existing_count > 0:
        batch_size = 500
        offset = 0
        while True:
            batch = metadata_collection.get(limit=batch_size, offset=offset)
            if not batch["ids"]:
                break
            meta_ids.extend(batch["ids"])
            offset += batch_size
    
    meta_ids = set(meta_ids)
    if not meta_ids:
        raise SystemExit(" No UUIDs found in metadata_embeddings. Cannot proceed with image embedding.")
    print(f" UUIDs available from metadata_embeddings: {len(meta_ids)}")
    
    return meta_ids
def _get_existing_image_ids(image_collection):
    # Get existing UUIDs (here ids) from image_embeddings collection.
    image_existing_count = image_collection.count()
    existing_image_ids = []
    
    if image_existing_count > 0:
        batch_size = 500
        offset = 0
        while True:
            batch = image_collection.get(limit=batch_size, offset=offset)
            if not batch["ids"]:
                break
            existing_image_ids.extend(batch["ids"])
            offset += batch_size
    
    existing_image_ids = set(existing_image_ids)
    print(f" Existing embeddings in image collection: {len(existing_image_ids)}")
    
    return existing_image_ids
def _find_candidate_images(client, meta_ids, existing_image_ids):
    # Find candidate images for embedding.
    
    trusted_zone = "trusted-zone"
    images_prefix = "images/"
    
    # List images in trusted-zone/images/
    trusted_objects = list(client.list_objects(trusted_zone, prefix=images_prefix, recursive=True))
    print(f" Scanned Trusted Zone images: found {len(trusted_objects)} files under '{images_prefix}'.")
    
    # Extract UUIDs and filter: must exist in metadata, not already embedded
    # Match the format of UUID anywhere in the object path (supports nested folders)
    uuid_re = re.compile(r"([a-f0-9\-]{36})\.\w+$", flags=re.I)
    candidates = []
    for obj in trusted_objects:
        m = uuid_re.search(obj.object_name)
        if not m:
            continue
        uid = m.group(1)
        if uid not in meta_ids:      # skip images not in metadata
            continue
        if uid in existing_image_ids:  # skip already embedded images
            continue
        candidates.append((uid, obj.object_name))
    
    print(f" Candidates for embedding: {len(candidates)} images (present in metadata, not yet embedded).")
    
    return candidates
def _process_image_embeddings(client, candidates, metadata_lookup, image_collection):
    """
    Process image embeddings using OpenCLIP and store them in ChromaDB.
    
    OpenCLIP (ViT-B-32, laion2b_s34b_b79k) is used to match the query models in 
    task scripts, ensuring dimension compatibility (512-dim embeddings).
    
    Why OpenCLIP for Exploitation Zone:
    - Model Consistency: Same model used for embedding and query ensures compatibility
    - Fine-Tuning Support: Embeddings work with both baseline and fine-tuned CLIP models
    - Multimodal Alignment: CLIP aligns image and text in the same embedding space
    - Local Inference: Works completely offline with no API dependencies
    - Task Integration: Seamless integration with downstream task scripts
    """
    image_collection_name = "image_embeddings"
    trusted_zone = "trusted-zone"
    
    if not candidates:
        print(" No new images to embed. Exiting.")
        return
    
    # =========================================================================
    # OpenCLIP Model Initialization
    # =========================================================================
    # We use ViT-B-32 with laion2b_s34b_b79k checkpoint (512-dim embeddings)
    # This matches the model used in task scripts for querying
    # =========================================================================
    print(f" Loading OpenCLIP model: ViT-B-32 (laion2b_s34b_b79k)")
    
    # Initialize OpenCLIP embeddings model
    clip_embeddings = OpenCLIPEmbeddings(
        model_name="ViT-B-32",
        checkpoint="laion2b_s34b_b79k"
    )
    
    print(f" OpenCLIP model loaded successfully")
    print(f" Embedding dimension: 512")

    uuids, docs, embeddings = [], [], []

    for idx, (uid, path) in enumerate(candidates):
        try:
            # Download image from MinIO
            data = client.get_object(trusted_zone, path)
            img_bytes = data.read()
            data.close()
            data.release_conn()

            # Save image to temporary file for OpenCLIP
            with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp_file:
                tmp_file.write(img_bytes)
                tmp_path = tmp_file.name
            
            # Generate embedding using OpenCLIP
            embedding = clip_embeddings.embed_image([tmp_path])[0]
            
            # Clean up temporary file
            os.unlink(tmp_path)

            if not embedding or len(embedding) == 0:
                print(f" No embedding generated for {path}")
                continue

            # Store data for insertion
            uuids.append(uid)
            docs.append(path)
            embeddings.append(embedding)
            
            # Progress logging (every 50 images)
            if (idx + 1) % 50 == 0:
                print(f" Processed {idx + 1}/{len(candidates)} images...")
            
        except Exception as e:
            print(f" Failed to process {path}: {e}")
            continue

    print(f" Successfully generated embeddings for {len(uuids)} images")

    # Add to ChromaDB along with enriched metadata
    if uuids and embeddings:
        # Create enriched metadata for each embedding
        enriched_metadatas = []
        for i, (uid, path) in enumerate(zip(uuids, docs)):
            # Get metadata for this UUID
            meta_info = metadata_lookup.get(uid, {})
            
            # Create enriched metadata
            enriched_meta = {
                "path": path,
                "species": meta_info.get('species', ''),
                "family": meta_info.get('family', ''),
                "class": meta_info.get('class', ''),
                "kingdom": meta_info.get('kingdom', ''),
                "scientific_name": meta_info.get('scientific_name', ''),
                "common": meta_info.get('common', ''),
                "genus": meta_info.get('genus', ''),
                "order": meta_info.get('order', ''),
                "phylum": meta_info.get('phylum', ''),
                "trusted_path": meta_info.get('trusted_path', ''),
                "embedding_model": "ViT-B-32/laion2b_s34b_b79k"  # Track which model generated this
            }
            enriched_metadatas.append(enriched_meta)
        
        image_collection.add(
            ids=uuids,
            embeddings=embeddings,
            metadatas=enriched_metadatas,
            documents=docs
        )
        print(f" Added {len(uuids)} new OpenCLIP image embeddings with enriched metadata to collection '{image_collection_name}'.")
    else:
        print(" No valid embeddings generated.")
def _display_final_summary(image_collection):
    # Display final summary of the processing.
    image_collection_name = "image_embeddings"
    
    # Summary
    total_count = image_collection.count()
    print(f"Collection '{image_collection_name}' now contains {total_count} total records.")
    print(" Exploitation Images processing complete.")
    print(" Note: Embeddings generated using OpenCLIP (ViT-B-32, laion2b_s34b_b79k)")


if __name__ == "__main__":
    process_exploitation_images()
    


 Loading metadata CSV for enrichment...
 Loading metadata: metadata/trusted_metadata_2025_12_30_23_00_23.csv
 Loaded metadata for 767 records
 UUIDs available from metadata_embeddings: 767
 Existing embeddings in image collection: 0
 Scanned Trusted Zone images: found 767 files under 'images/'.
 Candidates for embedding: 767 images (present in metadata, not yet embedded).
 Loading DINOv2 model: facebook/dinov2-base
 Using device: cpu


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

 DINOv2 model loaded successfully
 Embedding dimension: 768
 Processed 50/767 images...
 Processed 100/767 images...
 Processed 150/767 images...
 Processed 200/767 images...
 Processed 250/767 images...
 Processed 300/767 images...
 Processed 350/767 images...
 Processed 400/767 images...
 Processed 450/767 images...
 Processed 500/767 images...
 Processed 550/767 images...
 Processed 600/767 images...
 Processed 650/767 images...
 Processed 700/767 images...
 Processed 750/767 images...
 Successfully generated embeddings for 767 images
 Added 767 new DINOv2 image embeddings with enriched metadata to collection 'image_embeddings'.
Collection 'image_embeddings' now contains 767 total records.
 Exploitation Images processing complete.
 Note: Embeddings generated using DINOv2 (self-supervised vision transformer)
